In [1]:
import os, glob, zipfile, io
from google.colab import drive
drive.mount('/content/drive')

DRIVE = "/content/drive/MyDrive/fs2_bg_phone"
CP    = f"{DRIVE}/hifigan_finetune"          # your finetuned checkpoints

os.chdir("/content")
if not os.path.isdir("FastSpeech2"):
    os.system("git clone -b vocoder-finetune https://github.com/StankoI/FastSpeech2.git")
os.chdir("/content/FastSpeech2")
os.system("unzip -o -q hifigan/generator_universal.pth.tar.zip -d hifigan/")

# GTA mels from Drive
if not glob.glob("/content/FastSpeech2/gta_mels/Bulgarian/*.npy"):
    os.system(f"tar -C /content/FastSpeech2 -xf '{DRIVE}/gta_mels_Bulgarian.tar'")
GTA = "/content/FastSpeech2/gta_mels/Bulgarian"

# index original recordings inside the raw zip WITHOUT extracting all 44k files
RAW_ZIP   = zipfile.ZipFile(f"{DRIVE}/raw_data_Bulgarian.zip")
RAW_NAMES = {os.path.basename(n)[:-4]: n for n in RAW_ZIP.namelist() if n.endswith(".wav")}

print("GTA mels:", len(glob.glob(f"{GTA}/*.npy")))
print("checkpoints:", sorted(int(os.path.basename(p).split('_')[1]) for p in glob.glob(f"{CP}/g_*")))

Mounted at /content/drive
GTA mels: 44383
checkpoints: [0, 5000, 10000, 15000, 20000, 25000, 30000, 35000, 40000, 45000, 50000, 55000, 60000, 65000, 70000, 75000, 80000, 85000, 90000, 95000, 100000, 105000, 110000, 115000, 120000, 125000, 130000, 135000]


In [2]:
import sys, json, yaml, torch, numpy as np
os.chdir("/content/FastSpeech2"); sys.path.insert(0, "/content/FastSpeech2")
import hifigan

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pc = yaml.load(open("config/Bulgarian/preprocess.yaml"), Loader=yaml.FullLoader)
sr = pc["preprocessing"]["audio"]["sampling_rate"]; MAXV = pc["preprocessing"]["audio"]["max_wav_value"]
h  = hifigan.AttrDict(json.load(open("hifigan/config.json")))

def gen_from(state):
    g = hifigan.Generator(h).to(device); g.load_state_dict(state); g.eval(); g.remove_weight_norm(); return g

avail = sorted(int(os.path.basename(p).split("_")[1]) for p in glob.glob(f"{CP}/g_*"))
print("available steps:", avail)
STEPS_TO_TRY = [s for s in [50000, 70000, 90000, 12000, 135000 ] if s in avail]   # edit to taste

models = {}
with zipfile.ZipFile("hifigan/generator_universal.pth.tar.zip") as z:
    inner = [n for n in z.namelist() if n.endswith(".pth.tar")][0]
    models["universal"] = gen_from(torch.load(io.BytesIO(z.read(inner)), map_location=device)["generator"])
for s in STEPS_TO_TRY:
    models[f"ft_{s}"] = gen_from(torch.load(f"{CP}/g_{s:08d}", map_location=device)["generator"])
print("loaded:", list(models.keys()))

available steps: [0, 5000, 10000, 15000, 20000, 25000, 30000, 35000, 40000, 45000, 50000, 55000, 60000, 65000, 70000, 75000, 80000, 85000, 90000, 95000, 100000, 105000, 110000, 115000, 120000, 125000, 130000, 135000]


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Removing weight norm...
Removing weight norm...
Removing weight norm...
Removing weight norm...
Removing weight norm...
loaded: ['universal', 'ft_50000', 'ft_70000', 'ft_90000', 'ft_135000']


In [3]:
from IPython.display import Audio, display
from scipy.io import wavfile

def vocode(g, mel):
    with torch.no_grad(): return (g(mel).squeeze(1).cpu().numpy() * MAXV).astype("int16")[0]

for f in sorted(glob.glob(f"{GTA}/*.npy"))[:3]:        # change [:3] / pick specific ids as you like
    utt = os.path.splitext(os.path.basename(f))[0]
    mel = torch.from_numpy(np.load(f)).float().unsqueeze(0).to(device)
    print(f"================ {utt} ================")
    for name, g in models.items():
        print(name); display(Audio(vocode(g, mel), rate=sr))
    if utt in RAW_NAMES:
        _sr, d = wavfile.read(io.BytesIO(RAW_ZIP.read(RAW_NAMES[utt])))
        print("ORIGINAL recording"); display(Audio(d, rate=_sr))

================ gospodstvoto_na_bourne__01_pista01__0001 ================
universal


ft_50000


ft_70000


ft_90000


ft_135000


...

ft_50000


ft_70000


ft_90000


ft_135000


ORIGINAL recording


In [ ]:
import shutil
BEST_STEP = 50000   # set after listening
src = f"{CP}/g_{BEST_STEP:08d}"
shutil.copy(src, "/content/FastSpeech2/hifigan/generator_universal.pth.tar")  # used by synthesize.py
shutil.copy(src, f"{DRIVE}/generator_bg_finetuned.pth.tar")                    # Drive backup
print("locked in step", BEST_STEP)